# AMLGuard AI — Model Training

This notebook trains and compares machine learning models for
transaction-level fraud detection.

The feature engineering and data preparation were completed in
`02_feature_engineering.ipynb`.

The prepared datasets are stored in:

    data/processed/

The datasets are divided chronologically into:

- Training set: timestamps 0–139
- Validation set: timestamps 140–169
- Test set: timestamps 170–199

The training set is used to learn model patterns.

The validation set is used to compare models and tune their
configurations.

The test set is reserved for the final evaluation and should not
influence model selection.

Because fraudulent transactions represent only a very small portion
of the dataset, model performance will not be judged using accuracy
alone.

Important evaluation metrics will include:

- Precision
- Recall
- F1-score
- Average Precision / PR-AUC
- Confusion Matrix

The main objective is to identify fraudulent transactions while
carefully controlling false positives.

### 1.1 Import Required Libraries

We begin by importing the libraries required for loading the
prepared datasets, performing basic analysis, preprocessing the
features, and training machine learning models.

Model-specific libraries will be introduced later when the
corresponding models are trained.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


### 1.2 Define Project Paths

The processed datasets are stored outside the notebook so that
model training can be performed independently from the feature
engineering process.

We therefore define the project root and the location of the
processed datasets.

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

print("Processed data directory:")
print(PROCESSED_DATA)

Processed data directory:
c:\Users\MEGA\OneDrive - Higher Education Commission\Projects\AML_Guard\AML-Guard\data\processed


### 1.3 Load Prepared Datasets

The training, validation, and test datasets were generated by
`02_feature_engineering.ipynb`.

Each dataset contains:

- 29 candidate model features
- `IS_FRAUD` as the target variable

The datasets are loaded independently from disk rather than relying
on variables created in the previous notebook.

This makes the model-training notebook reproducible and prevents
the training process from depending on the previous notebook's
Python kernel state.

In [3]:
train_df = pd.read_csv(
    PROCESSED_DATA / "train.csv"
)

validation_df = pd.read_csv(
    PROCESSED_DATA / "validation.csv"
)

test_df = pd.read_csv(
    PROCESSED_DATA / "test.csv"
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (82328, 30)
Validation: (17677, 30)
Test: (17528, 30)


### 1.4 Initial Dataset Inspection

Before training any model, we verify that the prepared datasets
contain the expected columns and target distribution.

This step helps confirm that the data loaded by this notebook is
consistent with the output of the feature engineering pipeline.

In [4]:
print("Train columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

Train columns:
['TX_AMOUNT', 'TIMESTAMP', 'sender_initial_balance', 'sender_behavior_id', 'receiver_initial_balance', 'receiver_behavior_id', 'log_tx_amount', 'time_block', 'amount_to_sender_balance', 'sender_previous_tx_count', 'sender_previous_total_amount', 'sender_previous_avg_amount', 'receiver_previous_tx_count', 'receiver_previous_total_amount', 'receiver_previous_avg_amount', 'sender_tx_count_3step', 'sender_amount_3step', 'receiver_tx_count_3step', 'receiver_amount_3step', 'sender_time_since_last_tx', 'receiver_time_since_last_tx', 'is_new_sender_receiver_pair', 'sender_amount_vs_previous_avg', 'receiver_amount_vs_previous_avg', 'pair_previous_tx_count', 'pair_previous_total_amount', 'is_historical_cycle', 'sender_previous_unique_receivers', 'receiver_previous_unique_senders', 'IS_FRAUD']

Validation columns:
['TX_AMOUNT', 'TIMESTAMP', 'sender_initial_balance', 'sender_behavior_id', 'receiver_initial_balance', 'receiver_behavior_id', 'log_tx_amount', 'time_block', 'amount_to_s

In [5]:
print("Train fraud distribution:")
print(train_df["IS_FRAUD"].value_counts())

print("\nValidation fraud distribution:")
print(validation_df["IS_FRAUD"].value_counts())

print("\nTest fraud distribution:")
print(test_df["IS_FRAUD"].value_counts())

Train fraud distribution:
IS_FRAUD
False    82203
True       125
Name: count, dtype: int64

Validation fraud distribution:
IS_FRAUD
False    17652
True        25
Name: count, dtype: int64

Test fraud distribution:
IS_FRAUD
False    17503
True        25
Name: count, dtype: int64


### 1.5 Separate Features and Target

The `IS_FRAUD` column is the prediction target.

The remaining columns are the candidate model features.

We separate the target from the input features for each dataset.

The model will learn from the training features and training labels.

The validation and test labels are kept separate so that they can be
used for unbiased evaluation.

In [6]:
target_column = "IS_FRAUD"

X_train = train_df.drop(columns=[target_column])
y_train = train_df[target_column]

X_val = validation_df.drop(columns=[target_column])
y_val = validation_df[target_column]

X_test = test_df.drop(columns=[target_column])
y_test = test_df[target_column]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (82328, 29)
y_train: (82328,)

X_val: (17677, 29)
y_val: (17677,)

X_test: (17528, 29)
y_test: (17528,)


## 2. Class Imbalance

Fraudulent transactions are extremely rare in the dataset.

There are only 175 fraudulent transactions among 117,533 total
transactions.

This creates a severe class imbalance between:

- Legitimate transactions (`IS_FRAUD = 0`)
- Fraudulent transactions (`IS_FRAUD = 1`)

Class imbalance is important because a model can achieve very high
accuracy simply by predicting most transactions as legitimate.

For AML detection, this would not be a useful result.

Therefore, we will pay particular attention to minority-class
metrics such as:

- Precision
- Recall
- F1-score
- Average Precision / PR-AUC

Accuracy will still be reported for completeness, but it will not
be the primary measure of model quality.

In [7]:
# Calculate the class distribution in the training data.

train_class_counts = y_train.value_counts().sort_index()

train_class_percentages = (
    y_train.value_counts(normalize=True)
    .sort_index()
    * 100
)

print("Training class distribution:")
print(train_class_counts)

print("\nTraining class percentages:")
print(train_class_percentages)

Training class distribution:
IS_FRAUD
False    82203
True       125
Name: count, dtype: int64

Training class percentages:
IS_FRAUD
False    99.848168
True      0.151832
Name: proportion, dtype: float64


### 2.1 Class Distribution Across Dataset Splits

The fraud rate should be examined separately for the training,
validation, and test datasets.

Because the dataset was split chronologically rather than randomly,
the fraud rate does not need to be identical across the splits.

However, each split must contain enough fraudulent transactions to
evaluate model performance.

Our current split contains:

- 125 fraud cases in training
- 25 fraud cases in validation
- 25 fraud cases in testing

This allows us to train the models and perform separate validation
and final testing while preserving chronological order.

In [8]:
# Create a summary of class distribution across all dataset splits.

class_distribution = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Total": [
        len(y_train),
        len(y_val),
        len(y_test)
    ],
    "Fraud": [
        y_train.sum(),
        y_val.sum(),
        y_test.sum()
    ]
})

class_distribution["Non_Fraud"] = (
    class_distribution["Total"]
    - class_distribution["Fraud"]
)

class_distribution["Fraud_Rate_%"] = (
    class_distribution["Fraud"]
    / class_distribution["Total"]
    * 100
)

print(class_distribution)

        Split  Total  Fraud  Non_Fraud  Fraud_Rate_%
0       Train  82328    125      82203      0.151832
1  Validation  17677     25      17652      0.141427
2        Test  17528     25      17503      0.142629


### 2.2 Naive Baseline

Before training machine-learning models, we establish a naive
baseline.

The baseline predicts every transaction as non-fraudulent.

This is intentionally simple. Its purpose is not to create a useful
fraud detector, but to demonstrate why accuracy alone is misleading
for this highly imbalanced dataset.

If the baseline achieves very high accuracy while detecting no fraud,
a successful AML model must demonstrate substantially better
minority-class detection.

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Predict every validation transaction as non-fraud.
baseline_predictions = np.zeros(len(y_val), dtype=int)

print("Baseline Accuracy:",
      accuracy_score(y_val, baseline_predictions))

print("Baseline Precision:",
      precision_score(
          y_val,
          baseline_predictions,
          zero_division=0
      ))

print("Baseline Recall:",
      recall_score(
          y_val,
          baseline_predictions,
          zero_division=0
      ))

print("Baseline F1:",
      f1_score(
          y_val,
          baseline_predictions,
          zero_division=0
      ))

print("Baseline Average Precision:",
      average_precision_score(
          y_val,
          baseline_predictions
      ))

print("\nBaseline Confusion Matrix:")
print(confusion_matrix(y_val, baseline_predictions))

Baseline Accuracy: 0.998585732873225
Baseline Precision: 0.0
Baseline Recall: 0.0
Baseline F1: 0.0
Baseline Average Precision: 0.0014142671267749053

Baseline Confusion Matrix:
[[17652     0]
 [   25     0]]


### 3.1 Training Class Imbalance Ratio

The imbalance ratio shows how many legitimate transactions exist for
each fraudulent transaction in the training set.

This ratio helps us understand how severe the class imbalance is.

The ratio is calculated using only the training data because the
training data is the data used to learn model parameters.

In [10]:
# Count legitimate and fraudulent transactions in the training set.
negative_count = (y_train == False).sum()
positive_count = (y_train == True).sum()

# Calculate the ratio of legitimate transactions to fraudulent transactions.
imbalance_ratio = negative_count / positive_count

print("Legitimate transactions:", negative_count)
print("Fraudulent transactions:", positive_count)
print("Imbalance ratio:", imbalance_ratio)

Legitimate transactions: 82203
Fraudulent transactions: 125
Imbalance ratio: 657.624


### 3.2 Balanced Class Weights

A balanced weighting strategy assigns a larger weight to the minority
fraud class and a smaller weight to the majority legitimate class.

The weights are calculated from the training data only.

This does not change the original transactions. Instead, it changes
how strongly each class contributes to the model's learning process.

In [11]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([False, True])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

balanced_class_weights = dict(
    zip(classes, class_weights)
)

print("Balanced class weights:")
print(balanced_class_weights)

Balanced class weights:
{np.False_: np.float64(0.5007603128839581), np.True_: np.float64(329.312)}


### 3.3 Why We Are Not Oversampling Yet

Oversampling methods such as SMOTE can increase the number of
minority-class examples by creating additional synthetic samples.

However, our dataset contains only 125 fraudulent transactions in the
training period.

For the first modeling stage, we will avoid synthetic oversampling
and instead use class weighting.

Class weighting has several advantages for this project:

- It preserves the original training data.
- It does not create synthetic transactions.
- It is simple to reproduce.
- It is supported by several machine-learning models.
- It allows us to establish a clean baseline before introducing
  more advanced imbalance techniques.

If the initial models perform poorly, oversampling can be investigated
as a later experiment rather than being applied automatically.

## 4. Data Preprocessing

Before training Logistic Regression, the features must be prepared
according to their data types.

The two behavior ID features are categorical identifiers rather than
continuous numerical measurements. Therefore, they will be
one-hot encoded.

The remaining numerical features will be standardized so that their
scales are comparable.

The preprocessing pipeline will be fitted only on the training data.
The same fitted transformations will then be applied to the
validation and test datasets.

This prevents preprocessing leakage from validation or test data.

In [12]:
# Identify categorical features.
categorical_features = [
    "sender_behavior_id",
    "receiver_behavior_id"
]

# All remaining features are numerical.
numerical_features = [
    column for column in X_train.columns
    if column not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['sender_behavior_id', 'receiver_behavior_id']

Number of numerical features: 27

Numerical features:
['TX_AMOUNT', 'TIMESTAMP', 'sender_initial_balance', 'receiver_initial_balance', 'log_tx_amount', 'time_block', 'amount_to_sender_balance', 'sender_previous_tx_count', 'sender_previous_total_amount', 'sender_previous_avg_amount', 'receiver_previous_tx_count', 'receiver_previous_total_amount', 'receiver_previous_avg_amount', 'sender_tx_count_3step', 'sender_amount_3step', 'receiver_tx_count_3step', 'receiver_amount_3step', 'sender_time_since_last_tx', 'receiver_time_since_last_tx', 'is_new_sender_receiver_pair', 'sender_amount_vs_previous_avg', 'receiver_amount_vs_previous_avg', 'pair_previous_tx_count', 'pair_previous_total_amount', 'is_historical_cycle', 'sender_previous_unique_receivers', 'receiver_previous_unique_senders']


### 4.2 Preprocessing Pipeline

We will use a `ColumnTransformer` to apply different preprocessing
operations to different feature groups.

Categorical features:
- One-hot encoding

Numerical features:
- Standard scaling

The resulting transformer will be used inside a machine-learning
pipeline.

This ensures that the preprocessing is fitted using the training data
and consistently applied to validation and test data.

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


### 4.3 Fit Preprocessing on Training Data

The preprocessing transformer is fitted using only the training
features.

Validation and test data will not be used during this fitting step.

After transformation, the categorical behavior IDs will become
one-hot encoded columns and the numerical features will be
standardized.

In [14]:
# Fit the preprocessing transformer using training data only.
preprocessor.fit(X_train)

# Transform the training features.
X_train_processed = preprocessor.transform(X_train)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

Original training shape: (82328, 29)
Processed training shape: (82328, 37)


### 4.4 Transform Validation and Test Data

The preprocessing transformer has already been fitted using the
training data.

We now apply that same fitted transformer to the validation and test
datasets.

Only `transform()` is used for validation and test data.

This ensures that the model receives features prepared using the same
rules across all dataset splits while preventing information leakage.

In [15]:
# Transform validation and test features using the fitted preprocessor.
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (82328, 37)
Processed validation shape: (17677, 37)
Processed test shape: (17528, 37)


### 4.5 Preprocessing Verification

All three datasets should have the same number of processed features.

The number of rows must remain unchanged.

This check confirms that the preprocessing pipeline has been applied
consistently to training, validation, and test data.

In [16]:
print("Training rows:", X_train_processed.shape[0])
print("Validation rows:", X_val_processed.shape[0])
print("Test rows:", X_test_processed.shape[0])

print("\nProcessed feature count:")
print("Train:", X_train_processed.shape[1])
print("Validation:", X_val_processed.shape[1])
print("Test:", X_test_processed.shape[1])

print("\nFeature counts consistent:",
      X_train_processed.shape[1]
      == X_val_processed.shape[1]
      == X_test_processed.shape[1])

Training rows: 82328
Validation rows: 17677
Test rows: 17528

Processed feature count:
Train: 37
Validation: 37
Test: 37

Feature counts consistent: True


## 5. Logistic Regression

Logistic Regression is our first machine-learning model.

It is a useful baseline because it is relatively simple, fast to train,
and suitable for binary classification.

The model will use the processed features created in Section 4.

Because fraudulent transactions are extremely rare, balanced class
weights are used during training.

The model will learn from the training dataset only.

The validation dataset will be used afterward to evaluate how well the
trained model generalizes to unseen transactions.

In [17]:
from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model.
logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Train the model using the training data only.
logistic_model.fit(
    X_train_processed,
    y_train
)

print("Logistic Regression training completed successfully.")

Logistic Regression training completed successfully.


### 5.2 Generate Fraud Probabilities

The trained model can estimate the probability that each validation
transaction belongs to the fraud class.

These probabilities are important because AMLGuard will eventually need
to decide how much risk is acceptable before a transaction is flagged.

For the initial evaluation, we will later convert these probabilities
into binary predictions using a threshold of 0.50.

The probabilities themselves will also be useful for evaluating
Average Precision / PR-AUC.

In [18]:
# Generate the probability of the fraud class for each validation transaction.
y_val_probability = logistic_model.predict_proba(X_val_processed)[:, 1]

print("Number of validation probabilities:", len(y_val_probability))
print("Minimum probability:", y_val_probability.min())
print("Maximum probability:", y_val_probability.max())
print("Mean probability:", y_val_probability.mean())

Number of validation probabilities: 17677
Minimum probability: 1.425948498023306e-35
Maximum probability: 1.0
Mean probability: 0.03277649326106099


### 5.3 Convert Probabilities into Predictions

For the first Logistic Regression experiment, we use the standard
classification threshold of 0.50.

If the predicted fraud probability is at least 0.50, the transaction
is classified as fraudulent.

Otherwise, it is classified as legitimate.

This is only the initial threshold. We will later investigate whether
another threshold provides a better balance between fraud detection and
false positives using the validation data.

In [19]:
# Convert fraud probabilities into binary predictions using a 0.50 threshold.
y_val_pred = (y_val_probability >= 0.50).astype(int)

print("Predicted legitimate transactions:",
      (y_val_pred == 0).sum())

print("Predicted fraudulent transactions:",
      (y_val_pred == 1).sum())

Predicted legitimate transactions: 17263
Predicted fraudulent transactions: 414


### 5.4 Validation Performance

We now evaluate Logistic Regression on the validation dataset.

Because the dataset is highly imbalanced, accuracy alone is not enough.

We will examine:

- Accuracy
- Precision
- Recall
- F1-score
- Average Precision / PR-AUC
- Confusion Matrix

The Average Precision score will be calculated using the predicted
fraud probabilities rather than the binary predictions.

In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

logistic_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

logistic_precision = precision_score(
    y_val,
    y_val_pred,
    zero_division=0
)

logistic_recall = recall_score(
    y_val,
    y_val_pred,
    zero_division=0
)

logistic_f1 = f1_score(
    y_val,
    y_val_pred,
    zero_division=0
)

logistic_ap = average_precision_score(
    y_val,
    y_val_probability
)

logistic_cm = confusion_matrix(
    y_val,
    y_val_pred
)

print("Logistic Regression Validation Performance")
print("------------------------------------------")
print(f"Accuracy:          {logistic_accuracy:.6f}")
print(f"Precision:         {logistic_precision:.6f}")
print(f"Recall:            {logistic_recall:.6f}")
print(f"F1-score:          {logistic_f1:.6f}")
print(f"Average Precision: {logistic_ap:.6f}")

print("\nConfusion Matrix:")
print(logistic_cm)

Logistic Regression Validation Performance
------------------------------------------
Accuracy:          0.977994
Precision:         0.060386
Recall:            1.000000
F1-score:          0.113895
Average Precision: 0.229847

Confusion Matrix:
[[17263   389]
 [    0    25]]


## 6. Logistic Regression Threshold Analysis

Logistic Regression produces a fraud probability for every transaction.

The default classification threshold is 0.50:

- Probability >= 0.50 → Fraud
- Probability < 0.50 → Legitimate

However, the threshold can be changed.

A lower threshold generally makes the model more sensitive to possible
fraud, which can increase recall but also increase false positives.

A higher threshold generally makes the model more selective, which can
reduce false positives but may cause the model to miss some fraudulent
transactions.

For AMLGuard, this trade-off is important because both missed fraud and
unnecessary investigations have practical consequences.

The validation dataset will be used to study this trade-off.

The test dataset will remain untouched.


### 6.1 Compare Different Classification Thresholds

We will evaluate several thresholds using the validation probabilities
generated by Logistic Regression.

For each threshold, we will calculate:

- Precision
- Recall
- F1-score
- Number of predicted fraud transactions
- Number of false positives
- Number of fraud transactions detected

This allows us to see how the model's behavior changes as the threshold
changes.

In [21]:
thresholds = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

threshold_results = []

for threshold in thresholds:

    # Convert probabilities into binary predictions.
    predictions = (
        y_val_probability >= threshold
    ).astype(int)

    # Calculate confusion matrix values.
    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    # Calculate evaluation metrics.
    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        predictions,
        zero_division=0
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Predicted_Fraud": predictions.sum(),
        "False_Positives": fp,
        "Fraud_Detected": tp,
        "Fraud_Missed": fn
    })

threshold_results_df = pd.DataFrame(threshold_results)

print(threshold_results_df)

   Threshold  Precision  Recall        F1  Predicted_Fraud  False_Positives  \
0        0.1   0.022645     1.0  0.044287             1104             1079   
1        0.2   0.032982     1.0  0.063857              758              733   
2        0.3   0.041667     1.0  0.080000              600              575   
3        0.4   0.049900     1.0  0.095057              501              476   
4        0.5   0.060386     1.0  0.113895              414              389   
5        0.6   0.069444     1.0  0.129870              360              335   
6        0.7   0.085324     1.0  0.157233              293              268   
7        0.8   0.107296     1.0  0.193798              233              208   
8        0.9   0.142857     1.0  0.250000              175              150   

   Fraud_Detected  Fraud_Missed  
0              25             0  
1              25             0  
2              25             0  
3              25             0  
4              25             0  
5    

### 6.2 Threshold Comparison

The threshold results are easier to interpret when the main metrics
are displayed with consistent decimal precision.

Precision, recall, and F1-score are shown as percentages.

The number of detected and missed fraud transactions is shown as
counts.

In [22]:
threshold_display = threshold_results_df.copy()

threshold_display["Precision"] = (
    threshold_display["Precision"] * 100
).round(2)

threshold_display["Recall"] = (
    threshold_display["Recall"] * 100
).round(2)

threshold_display["F1"] = (
    threshold_display["F1"] * 100
).round(2)

print(threshold_display.to_string(index=False))

 Threshold  Precision  Recall    F1  Predicted_Fraud  False_Positives  Fraud_Detected  Fraud_Missed
       0.1       2.26   100.0  4.43             1104             1079              25             0
       0.2       3.30   100.0  6.39              758              733              25             0
       0.3       4.17   100.0  8.00              600              575              25             0
       0.4       4.99   100.0  9.51              501              476              25             0
       0.5       6.04   100.0 11.39              414              389              25             0
       0.6       6.94   100.0 12.99              360              335              25             0
       0.7       8.53   100.0 15.72              293              268              25             0
       0.8      10.73   100.0 19.38              233              208              25             0
       0.9      14.29   100.0 25.00              175              150              25             0


### 6.3 Best Validation Threshold by F1-score

F1-score balances precision and recall.

We will identify the threshold that produces the highest F1-score on
the validation dataset.

This threshold is a validation-based candidate only. It will not be
used to evaluate the test set until the model-selection stage is
complete.

In [23]:
best_f1_row = threshold_results_df.loc[
    threshold_results_df["F1"].idxmax()
]

print("Best threshold by validation F1:")
print(f"Threshold: {best_f1_row['Threshold']}")
print(f"Precision: {best_f1_row['Precision']:.6f}")
print(f"Recall: {best_f1_row['Recall']:.6f}")
print(f"F1-score: {best_f1_row['F1']:.6f}")
print(f"Predicted fraud: {best_f1_row['Predicted_Fraud']}")
print(f"False positives: {best_f1_row['False_Positives']}")
print(f"Fraud detected: {best_f1_row['Fraud_Detected']}")
print(f"Fraud missed: {best_f1_row['Fraud_Missed']}")

Best threshold by validation F1:
Threshold: 0.9
Precision: 0.142857
Recall: 1.000000
F1-score: 0.250000
Predicted fraud: 175.0
False positives: 150.0
Fraud detected: 25.0
Fraud missed: 0.0


### 6.4 Store Logistic Regression Threshold Results

The threshold analysis results are stored so that they can be
compared with the validation results of other machine-learning models.

The validation results will be used for model selection and threshold
selection.

The test dataset will remain untouched until the final evaluation.

In [24]:
# Store the Logistic Regression threshold results.
logistic_threshold_results = threshold_results_df.copy()

print("Logistic Regression threshold results stored.")
print(
    logistic_threshold_results[
        [
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "Predicted_Fraud",
            "False_Positives",
            "Fraud_Detected",
            "Fraud_Missed"
        ]
    ]
)

Logistic Regression threshold results stored.
   Threshold  Precision  Recall        F1  Predicted_Fraud  False_Positives  \
0        0.1   0.022645     1.0  0.044287             1104             1079   
1        0.2   0.032982     1.0  0.063857              758              733   
2        0.3   0.041667     1.0  0.080000              600              575   
3        0.4   0.049900     1.0  0.095057              501              476   
4        0.5   0.060386     1.0  0.113895              414              389   
5        0.6   0.069444     1.0  0.129870              360              335   
6        0.7   0.085324     1.0  0.157233              293              268   
7        0.8   0.107296     1.0  0.193798              233              208   
8        0.9   0.142857     1.0  0.250000              175              150   

   Fraud_Detected  Fraud_Missed  
0              25             0  
1              25             0  
2              25             0  
3              25          

## 7. Decision Tree

The Decision Tree is our second machine-learning model.

Unlike Logistic Regression, a Decision Tree can learn non-linear
relationships between features.

The model can recursively split transactions according to feature
values and learn combinations of conditions associated with fraud.

The model will use balanced class weights because fraudulent
transactions are extremely rare.

The model will be trained using the training dataset only.

The validation dataset will be used for evaluation and threshold
analysis.

In [25]:
from sklearn.tree import DecisionTreeClassifier

# Create the Decision Tree model.
decision_tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42,
    max_depth=6
)

# Train the model.
decision_tree_model.fit(
    X_train_processed,
    y_train
)

print("Decision Tree training completed successfully.")

Decision Tree training completed successfully.


### 7.2 Generate Decision Tree Validation Probabilities

The trained Decision Tree will produce a fraud probability for every
validation transaction.

These probabilities allow us to evaluate the model using the same
threshold-based approach used for Logistic Regression.

In [26]:
# Generate fraud probabilities for the validation dataset.
dt_val_probability = decision_tree_model.predict_proba(
    X_val_processed
)[:, 1]

print("Number of validation probabilities:", len(dt_val_probability))
print("Minimum probability:", dt_val_probability.min())
print("Maximum probability:", dt_val_probability.max())
print("Mean probability:", dt_val_probability.mean())

Number of validation probabilities: 17677
Minimum probability: 0.0
Maximum probability: 1.0
Mean probability: 0.0014142671267749053


### 7.3 Decision Tree Validation Performance

The Decision Tree is initially evaluated using a classification
threshold of 0.50.

The same evaluation metrics used for Logistic Regression will be
reported so that the models can be compared fairly.

In [27]:
# Convert probabilities into predictions using a 0.50 threshold.
dt_val_pred = (
    dt_val_probability >= 0.50
).astype(int)

# Calculate evaluation metrics.
dt_accuracy = accuracy_score(
    y_val,
    dt_val_pred
)

dt_precision = precision_score(
    y_val,
    dt_val_pred,
    zero_division=0
)

dt_recall = recall_score(
    y_val,
    dt_val_pred,
    zero_division=0
)

dt_f1 = f1_score(
    y_val,
    dt_val_pred,
    zero_division=0
)

dt_ap = average_precision_score(
    y_val,
    dt_val_probability
)

dt_cm = confusion_matrix(
    y_val,
    dt_val_pred
)

print("Decision Tree Validation Performance")
print("------------------------------------")
print(f"Accuracy:          {dt_accuracy:.6f}")
print(f"Precision:         {dt_precision:.6f}")
print(f"Recall:            {dt_recall:.6f}")
print(f"F1-score:          {dt_f1:.6f}")
print(f"Average Precision: {dt_ap:.6f}")

print("\nConfusion Matrix:")
print(dt_cm)

Decision Tree Validation Performance
------------------------------------
Accuracy:          1.000000
Precision:         1.000000
Recall:            1.000000
F1-score:          1.000000
Average Precision: 1.000000

Confusion Matrix:
[[17652     0]
 [    0    25]]


### 7.4 Decision Tree Inspection

The Decision Tree achieved perfect validation performance.

Before moving to a more complex model, we inspect the learned tree
structure to understand which features are being used for its
decisions.

Feature importance values indicate how much each feature contributed
to reducing impurity during tree construction.

These values should be interpreted as model-specific indicators rather
than proof that a feature is causally responsible for fraud.

In [28]:
# Get feature names after preprocessing.
feature_names = preprocessor.get_feature_names_out()

# Get feature importance values from the trained Decision Tree.
dt_feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": decision_tree_model.feature_importances_
})

# Sort features by importance.
dt_feature_importance = dt_feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("Decision Tree Feature Importance:")
print(dt_feature_importance.to_string(index=False))

Decision Tree Feature Importance:
                              Feature   Importance
   num__sender_amount_vs_previous_avg 8.662549e-01
                   num__log_tx_amount 7.282462e-02
           num__sender_tx_count_3step 3.490413e-02
                       num__TX_AMOUNT 1.855568e-02
        num__sender_previous_tx_count 7.339268e-03
          num__sender_initial_balance 1.214193e-04
      num__receiver_previous_tx_count 1.243063e-15
 num__receiver_amount_vs_previous_avg 5.090025e-16
          cat__receiver_behavior_id_1 0.000000e+00
            cat__sender_behavior_id_5 0.000000e+00
          cat__receiver_behavior_id_2 0.000000e+00
          num__pair_previous_tx_count 0.000000e+00
            cat__sender_behavior_id_3 0.000000e+00
          cat__receiver_behavior_id_3 0.000000e+00
            cat__sender_behavior_id_2 0.000000e+00
            cat__sender_behavior_id_1 0.000000e+00
num__receiver_previous_unique_senders 0.000000e+00
num__sender_previous_unique_receivers 0.000000e+

### 7.5 Training Performance Check

The Decision Tree achieved perfect validation performance.

We will also evaluate its performance on the training data.

Comparing training and validation performance helps us identify
possible overfitting.

A large difference between training and validation performance would
suggest that the model may be memorizing the training data.


In [29]:
# Generate training probabilities and predictions.
dt_train_probability = decision_tree_model.predict_proba(
    X_train_processed
)[:, 1]

dt_train_pred = (
    dt_train_probability >= 0.50
).astype(int)

# Calculate training metrics.
dt_train_accuracy = accuracy_score(
    y_train,
    dt_train_pred
)

dt_train_precision = precision_score(
    y_train,
    dt_train_pred,
    zero_division=0
)

dt_train_recall = recall_score(
    y_train,
    dt_train_pred,
    zero_division=0
)

dt_train_f1 = f1_score(
    y_train,
    dt_train_pred,
    zero_division=0
)

dt_train_ap = average_precision_score(
    y_train,
    dt_train_probability
)

print("Decision Tree Training Performance")
print("-----------------------------------")
print(f"Accuracy:          {dt_train_accuracy:.6f}")
print(f"Precision:         {dt_train_precision:.6f}")
print(f"Recall:            {dt_train_recall:.6f}")
print(f"F1-score:          {dt_train_f1:.6f}")
print(f"Average Precision: {dt_train_ap:.6f}")

Decision Tree Training Performance
-----------------------------------
Accuracy:          1.000000
Precision:         1.000000
Recall:            1.000000
F1-score:          1.000000
Average Precision: 1.000000


### 7.6 Dominant Feature Inspection

The Decision Tree feature importance analysis showed that
`sender_amount_vs_previous_avg` is responsible for most of the
model's predictive power.

Before continuing with other models, we inspect this feature
separately for fraudulent and legitimate transactions.

This helps us understand whether the model is learning a meaningful
historical-behavior pattern or simply exploiting an unusual property
of the synthetic dataset.

The comparison includes:

- minimum value
- median value
- mean value
- maximum value
- number of missing values

The target label is used only for analysis here and is not used as
an input feature.

In [30]:
dominant_feature = "sender_amount_vs_previous_avg"

feature_analysis = pd.DataFrame({
    "Class": ["Legitimate", "Fraud"],
    "Min": [
        X_train.loc[y_train == False, dominant_feature].min(),
        X_train.loc[y_train == True, dominant_feature].min()
    ],
    "Median": [
        X_train.loc[y_train == False, dominant_feature].median(),
        X_train.loc[y_train == True, dominant_feature].median()
    ],
    "Mean": [
        X_train.loc[y_train == False, dominant_feature].mean(),
        X_train.loc[y_train == True, dominant_feature].mean()
    ],
    "Max": [
        X_train.loc[y_train == False, dominant_feature].max(),
        X_train.loc[y_train == True, dominant_feature].max()
    ],
    "Missing": [
        X_train.loc[y_train == False, dominant_feature].isna().sum(),
        X_train.loc[y_train == True, dominant_feature].isna().sum()
    ]
})

print("Dominant Feature Analysis:")
print(feature_analysis.to_string(index=False))

Dominant Feature Analysis:
     Class  Min   Median     Mean          Max  Missing
Legitimate  0.0 1.000000 1.182933 10032.397260        0
     Fraud  0.0 0.028794 0.150540     2.458128        0


### 7.7 Fraud and Legitimate Value Ranges

We now check how much the value ranges of the dominant feature
overlap between fraudulent and legitimate transactions.

A strong separation would explain why the Decision Tree can achieve
perfect validation performance.

This is an exploratory analysis only and does not modify the model.

In [31]:
fraud_values = X_train.loc[
    y_train == True,
    dominant_feature
]

legitimate_values = X_train.loc[
    y_train == False,
    dominant_feature
]

print("Legitimate transactions:")
print(f"Minimum: {legitimate_values.min():.6f}")
print(f"Maximum: {legitimate_values.max():.6f}")

print("\nFraudulent transactions:")
print(f"Minimum: {fraud_values.min():.6f}")
print(f"Maximum: {fraud_values.max():.6f}")

Legitimate transactions:
Minimum: 0.000000
Maximum: 10032.397260

Fraudulent transactions:
Minimum: 0.000000
Maximum: 2.458128


### 8.1 Random Forest

Random Forest is an ensemble learning method that combines the
predictions of multiple Decision Trees.

Each tree learns different patterns from the training data, and the
forest combines their predictions to produce the final result.

We use Random Forest because it can capture nonlinear relationships
between transaction behavior and fraud.

The model uses:

- 200 trees
- balanced class weights
- maximum tree depth of 10
- minimum 2 samples per leaf
- all available CPU cores

The validation dataset remains untouched during training.

In [32]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

print("Random Forest training completed successfully.")

Random Forest training completed successfully.


### 8.3 Random Forest Validation Performance

The Random Forest has been trained using only the training dataset.

We now generate fraud probabilities for the validation dataset and
evaluate the model using the same metrics used for the previous models.

The test dataset is still untouched.

Evaluation metrics:

- Accuracy
- Precision
- Recall
- F1-score
- Average Precision
- Confusion Matrix

In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

rf_val_probability = random_forest_model.predict_proba(
    X_val_processed
)[:, 1]

rf_val_pred = (
    rf_val_probability >= 0.50
).astype(int)

rf_val_accuracy = accuracy_score(
    y_val,
    rf_val_pred
)

rf_val_precision = precision_score(
    y_val,
    rf_val_pred,
    zero_division=0
)

rf_val_recall = recall_score(
    y_val,
    rf_val_pred,
    zero_division=0
)

rf_val_f1 = f1_score(
    y_val,
    rf_val_pred,
    zero_division=0
)

rf_val_ap = average_precision_score(
    y_val,
    rf_val_probability
)

rf_val_cm = confusion_matrix(
    y_val,
    rf_val_pred
)

print("Random Forest Validation Performance")
print("-------------------------------------")
print(f"Accuracy:          {rf_val_accuracy:.6f}")
print(f"Precision:         {rf_val_precision:.6f}")
print(f"Recall:            {rf_val_recall:.6f}")
print(f"F1-score:          {rf_val_f1:.6f}")
print(f"Average Precision: {rf_val_ap:.6f}")

print("\nConfusion Matrix:")
print(rf_val_cm)

Random Forest Validation Performance
-------------------------------------
Accuracy:          1.000000
Precision:         1.000000
Recall:            1.000000
F1-score:          1.000000
Average Precision: 1.000000

Confusion Matrix:
[[17652     0]
 [    0    25]]


### 8.4 Random Forest Feature Importance

The Random Forest also achieved perfect validation performance.

We now inspect its feature importance values and compare them with
the Decision Tree.

If both tree-based models rely heavily on the same features, this
provides additional evidence that the strong validation performance
is driven by patterns present in the dataset rather than by one
specific tree structure.

Feature importance indicates how useful each feature was for the
Random Forest's tree-based splits. It does not establish causal
relationships.

In [34]:
rf_feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": random_forest_model.feature_importances_
})

rf_feature_importance = rf_feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("Random Forest Feature Importance:")
print(rf_feature_importance.to_string(index=False))

Random Forest Feature Importance:
                              Feature  Importance
   num__sender_amount_vs_previous_avg    0.214993
        num__amount_to_sender_balance    0.197478
                       num__TX_AMOUNT    0.165106
                   num__log_tx_amount    0.136029
 num__receiver_amount_vs_previous_avg    0.060704
      num__pair_previous_total_amount    0.041519
     num__is_new_sender_receiver_pair    0.031447
          num__pair_previous_tx_count    0.027915
      num__sender_previous_avg_amount    0.021790
             num__sender_amount_3step    0.017980
    num__sender_previous_total_amount    0.010077
            cat__sender_behavior_id_1    0.009097
       num__sender_time_since_last_tx    0.008820
          num__sender_initial_balance    0.007987
num__sender_previous_unique_receivers    0.007326
           num__sender_tx_count_3step    0.005899
        num__sender_previous_tx_count    0.005063
            cat__sender_behavior_id_5    0.004399
    num__receive

# Section 9 — XGBoost

XGBoost is a gradient-boosting algorithm that builds decision trees
sequentially.

Unlike Random Forest, where many trees are trained independently,
XGBoost trains each new tree to improve the errors made by previous
trees.

This makes XGBoost a strong candidate for structured/tabular
classification problems.

Because the fraud class is highly imbalanced, we use
`scale_pos_weight` calculated from the training data.

The validation and test datasets remain untouched during training.

In [35]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


### 9.2 XGBoost Class Imbalance

XGBoost handles class imbalance using `scale_pos_weight`.

The value is calculated from the training dataset as:

    number of legitimate transactions
    -----------------------------------
       number of fraudulent transactions

This gives greater importance to fraudulent transactions during
training.

Only the training labels are used to calculate this value.

In [36]:
# Calculate the class imbalance ratio from the training data.
xgb_scale_pos_weight = (
    (y_train == False).sum()
    / (y_train == True).sum()
)

print("XGBoost scale_pos_weight:", xgb_scale_pos_weight)

XGBoost scale_pos_weight: 657.624


### 9.3 Restore Training Target

The XGBoost training error was caused by the `y_train` variable
being overwritten earlier in the notebook.

The processed training features still contain 82,328 rows, but
`y_train` contained only 120 values.

We restore the training target directly from the original processed
training dataset to ensure that the features and labels have matching
row counts.

In [37]:
# Restore the training target from the processed training dataset.
y_train = train_df["IS_FRAUD"].to_numpy()

print("X_train_processed shape:", X_train_processed.shape)
print("y_train shape:", y_train.shape)
print("y_train type:", type(y_train))
print("y_train dtype:", y_train.dtype)

print("\nTarget values and counts:")
unique_values, counts = np.unique(y_train, return_counts=True)

for value, count in zip(unique_values, counts):
    print(f"{value}: {count}")

X_train_processed shape: (82328, 37)
y_train shape: (82328,)
y_train type: <class 'numpy.ndarray'>
y_train dtype: bool

Target values and counts:
False: 82203
True: 125


In [38]:
print("Training target:", y_train.shape)
print("Validation target:", y_val.shape)
print("Test target:", y_test.shape)

print("\nExpected:")
print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Training target: (82328,)
Validation target: (17677,)
Test target: (17528,)

Expected:
Training: 82328
Validation: 17677
Test: 17528


In [39]:
print("xgb_model exists:", "xgb_model" in globals())

if "xgb_model" in globals():
    print(type(xgb_model))

xgb_model exists: False


In [40]:
# Train the XGBoost model.

import xgboost as xgb

xgb_scale_pos_weight = (
    (y_train == False).sum()
    / (y_train == True).sum()
)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=xgb_scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_processed,
    y_train
)

print("XGBoost model created and trained successfully.")
print("Model type:", type(xgb_model))

XGBoost model created and trained successfully.
Model type: <class 'xgboost.sklearn.XGBClassifier'>


### 9.4 XGBoost Validation Performance

The XGBoost model has been trained using the training dataset.

We now evaluate its predictions on the validation dataset using the
same metrics used for the previous models.

The test dataset remains untouched and will only be used after the
final model and decision threshold have been selected.

In [41]:
# Generate fraud probabilities for the validation dataset.
xgb_val_probability = xgb_model.predict_proba(
    X_val_processed
)[:, 1]

# Convert probabilities into class predictions using the default threshold.
xgb_val_pred = (
    xgb_val_probability >= 0.50
).astype(int)

# Calculate validation metrics.
xgb_val_accuracy = accuracy_score(
    y_val,
    xgb_val_pred
)

xgb_val_precision = precision_score(
    y_val,
    xgb_val_pred,
    zero_division=0
)

xgb_val_recall = recall_score(
    y_val,
    xgb_val_pred,
    zero_division=0
)

xgb_val_f1 = f1_score(
    y_val,
    xgb_val_pred,
    zero_division=0
)

xgb_val_ap = average_precision_score(
    y_val,
    xgb_val_probability
)

# Generate confusion matrix.
xgb_val_cm = confusion_matrix(
    y_val,
    xgb_val_pred
)

print("XGBoost Validation Performance")
print("--------------------------------")
print(f"Accuracy:          {xgb_val_accuracy:.6f}")
print(f"Precision:         {xgb_val_precision:.6f}")
print(f"Recall:            {xgb_val_recall:.6f}")
print(f"F1-score:          {xgb_val_f1:.6f}")
print(f"Average Precision: {xgb_val_ap:.6f}")

print("\nConfusion Matrix:")
print(xgb_val_cm)

XGBoost Validation Performance
--------------------------------
Accuracy:          1.000000
Precision:         1.000000
Recall:            1.000000
F1-score:          1.000000
Average Precision: 1.000000

Confusion Matrix:
[[17652     0]
 [    0    25]]


# Section 10 — Model Comparison

The four trained models are now evaluated on the same validation
dataset.

We compare their performance using precision, recall, F1-score, and
Average Precision.

The validation set is used for model selection.

The test set remains untouched until the final model has been selected.

In [42]:
# Create a summary of validation performance for all trained models.
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        0.977994,
        1.000000,
        1.000000,
        xgb_val_accuracy
    ],
    "Precision": [
        0.060386,
        1.000000,
        1.000000,
        xgb_val_precision
    ],
    "Recall": [
        1.000000,
        1.000000,
        1.000000,
        xgb_val_recall
    ],
    "F1": [
        0.113895,
        1.000000,
        1.000000,
        xgb_val_f1
    ],
    "Average_Precision": [
        0.229847,
        1.000000,
        1.000000,
        xgb_val_ap
    ]
})

print("Validation Model Comparison:")
print(model_comparison.to_string(index=False))

Validation Model Comparison:
              Model  Accuracy  Precision  Recall       F1  Average_Precision
Logistic Regression  0.977994   0.060386     1.0 0.113895           0.229847
      Decision Tree  1.000000   1.000000     1.0 1.000000           1.000000
      Random Forest  1.000000   1.000000     1.0 1.000000           1.000000
            XGBoost  1.000000   1.000000     1.0 1.000000           1.000000


# Section 10 — Model Comparison

The four trained models are now evaluated on the same validation
dataset.

We compare their performance using precision, recall, F1-score, and
Average Precision.

The validation set is used for model selection.

The test set remains untouched until the final model has been selected.

In [43]:
# Create a summary of validation performance for all trained models.
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        0.977994,
        1.000000,
        1.000000,
        xgb_val_accuracy
    ],
    "Precision": [
        0.060386,
        1.000000,
        1.000000,
        xgb_val_precision
    ],
    "Recall": [
        1.000000,
        1.000000,
        1.000000,
        xgb_val_recall
    ],
    "F1": [
        0.113895,
        1.000000,
        1.000000,
        xgb_val_f1
    ],
    "Average_Precision": [
        0.229847,
        1.000000,
        1.000000,
        xgb_val_ap
    ]
})

print("Validation Model Comparison:")
print(model_comparison.to_string(index=False))

Validation Model Comparison:
              Model  Accuracy  Precision  Recall       F1  Average_Precision
Logistic Regression  0.977994   0.060386     1.0 0.113895           0.229847
      Decision Tree  1.000000   1.000000     1.0 1.000000           1.000000
      Random Forest  1.000000   1.000000     1.0 1.000000           1.000000
            XGBoost  1.000000   1.000000     1.0 1.000000           1.000000


### 10.1 Model Selection

The model with the strongest validation performance will be selected
for final evaluation.

Average Precision and F1-score are prioritized because the dataset has
severe class imbalance.

The test dataset must not influence this selection.

In [44]:
# Select the model with the highest validation Average Precision.
best_model_name = model_comparison.loc[
    model_comparison["Average_Precision"].idxmax(),
    "Model"
]

print("Selected model:", best_model_name)

Selected model: Decision Tree


# Section 11 — Final Model Selection

XGBoost achieved the same perfect validation performance as the
Decision Tree and Random Forest.

Because XGBoost is already trained and provides a strong ensemble-based
model for the final system, it will be used as the selected model.

The test dataset has not been used for model selection.

The selected model will now be evaluated once on the untouched test set.

In [45]:
# Use XGBoost as the final selected model.
final_model = xgb_model

# Use the default classification threshold of 0.50.
final_threshold = 0.50

print("Final selected model: XGBoost")
print("Final classification threshold:", final_threshold)

Final selected model: XGBoost
Final classification threshold: 0.5


# Section 12 — Final Test Evaluation

The XGBoost model has been selected using the validation dataset.

The test dataset has remained untouched during model selection.

The final model is now evaluated on the test dataset using:

- Accuracy
- Precision
- Recall
- F1-score
- Average Precision
- Confusion Matrix

The test results represent the final performance of the selected model
on previously unseen transactions.

In [46]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Generate fraud probabilities for the untouched test set.
final_test_probability = final_model.predict_proba(X_test_processed)[:, 1]

# Convert probabilities into binary predictions using the selected threshold.
final_test_prediction = (
    final_test_probability >= final_threshold
).astype(int)

# Calculate final test metrics.
final_test_accuracy = accuracy_score(
    y_test,
    final_test_prediction
)

final_test_precision = precision_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_recall = recall_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_f1 = f1_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_ap = average_precision_score(
    y_test,
    final_test_probability
)

final_test_cm = confusion_matrix(
    y_test,
    final_test_prediction
)

print("Final XGBoost Test Performance")
print("--------------------------------")
print(f"Accuracy:          {final_test_accuracy:.6f}")
print(f"Precision:         {final_test_precision:.6f}")
print(f"Recall:            {final_test_recall:.6f}")
print(f"F1-score:          {final_test_f1:.6f}")
print(f"Average Precision: {final_test_ap:.6f}")

print("\nConfusion Matrix:")
print(final_test_cm)

Final XGBoost Test Performance
--------------------------------
Accuracy:          1.000000
Precision:         1.000000
Recall:            1.000000
F1-score:          1.000000
Average Precision: 1.000000

Confusion Matrix:
[[17503     0]
 [    0    25]]


# Section 13 — Final Model Results

The final XGBoost model was selected using the validation dataset and
then evaluated on the untouched test dataset.

The model achieved perfect performance on the test set:

- Accuracy: 100%
- Precision: 100%
- Recall: 100%
- F1-score: 100%
- Average Precision: 100%

The confusion matrix shows:

- True Negatives: 17,503
- False Positives: 0
- False Negatives: 0
- True Positives: 25

These results indicate that the model successfully detected all
fraudulent transactions in the test dataset without incorrectly
flagging legitimate transactions.

Because the dataset is synthetic and contains only 25 fraudulent
transactions in the test set, these results should not be interpreted
as evidence of real-world performance. Further evaluation on
realistic and more diverse financial transaction data would be
required before deployment.

In [47]:
# Create a compact summary of the final model performance.
final_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Average Precision"
    ],
    "Test Score": [
        final_test_accuracy,
        final_test_precision,
        final_test_recall,
        final_test_f1,
        final_test_ap
    ]
})

print("Final XGBoost Test Results:")
print(final_results.to_string(index=False))

Final XGBoost Test Results:
           Metric  Test Score
         Accuracy         1.0
        Precision         1.0
           Recall         1.0
         F1-score         1.0
Average Precision         1.0


# Section 14 — Model Training Completed

The AMLGuard model training pipeline has been completed successfully.

The following models were trained and evaluated:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

The models were compared using validation data, with particular
attention to Precision, Recall, F1-score, and Average Precision because
of the severe class imbalance.

XGBoost was selected as the final model.

Final test performance:

- Accuracy: 100%
- Precision: 100%
- Recall: 100%
- F1-score: 100%
- Average Precision: 100%

The final test set contained 17,528 transactions, including 25
fraudulent transactions.

The model correctly classified all test transactions.

Important limitation:
The dataset is synthetic and highly imbalanced. Therefore, the
perfect test performance should not be interpreted as guaranteed
real-world performance. Evaluation on larger, realistic financial
datasets would be required before deployment.

Next steps may include model persistence, deployment, and integration
with the AMLGuard application.

In [48]:
print("AMLGuard model training completed successfully.")
print("Final model: XGBoost")
print(f"Test F1-score: {final_test_f1:.4f}")
print(f"Test Average Precision: {final_test_ap:.4f}")

AMLGuard model training completed successfully.
Final model: XGBoost
Test F1-score: 1.0000
Test Average Precision: 1.0000
